# 15. Dunder Methods & Slots (5+ Years Interview Guide)
Deep architectural analysis of Python data model special methods, memory optimization via __slots__, string representations (__str__ vs __repr__), operator overloading, and custom container protocols.

### Key 5-Year Interview Concepts Covered:
- **Memory Optimization with `__slots__`**: Eliminating `__dict__` overhead, reducing memory by 40-60% for millions of objects.
- **String Protocols (`__str__` vs `__repr__`)**: User-facing readability vs unambiguous developer debugging representation.
- **Container & Sequence Protocols**: `__len__`, `__getitem__`, `__setitem__`, and `__contains__`.
- **Operator Overloading & Hashing**: Arithmetic dunders, rich comparisons (`__eq__`, `__lt__`), and `__hash__` consistency rules.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Memory Optimization with `__slots__`
**Explanation**: By default, every Python instance stores attributes in a dynamic `__dict__` dictionary, consuming substantial memory overhead. Defining `__slots__ = ('id', 'amount')` tells CPython to allocate a fixed-size array of attribute descriptors instead of `__dict__`. This reduces memory consumption by 40-60% when instantiating millions of domain objects.

**Syntax**: `class Record: __slots__ = ('tx_id', 'amount')`

In [ ]:
class SlottedCoordinates:
    __slots__ = ['coordinate_x']
    def __init__(self): self.coordinate_x = 1
print(SlottedCoordinates().coordinate_x)

### 2. Slot Unbinding & Dynamic Attribute Restrictions
**Explanation**: When `__slots__` is defined without `'__dict__'`, instances CANNOT have arbitrary dynamic attributes added at runtime (`obj.unlisted_field = 1` raises `AttributeError`). Subclasses also do not inherit slots automatically; they must declare their own `__slots__` to maintain memory optimization.

**Syntax**: `obj.unlisted_attr = val  # Raises AttributeError`

In [ ]:
class SlottedCoordinates:
    __slots__ = ['coordinate_x']
try:
    print(SlottedCoordinates().__dict__)
except AttributeError as error_message:
    print('No dict namespace:', error_message)

### 3. Memory Footprint Benchmarking: `__dict__` vs `__slots__`
**Explanation**: Inspecting instances with `sys.getsizeof()` reveals that slotted instances bypass the dictionary hash table allocation entirely. In large fintech caching or batch processing architectures, using slots saves gigabytes of RAM across transaction queues.

**Syntax**: `sys.getsizeof(slotted_obj) < sys.getsizeof(dict_obj)`

In [ ]:
import sys
class StandardCoordinatesDictionary: pass
class SlottedCoordinates: __slots__ = ['coordinate_x']
coordinates_dict_instance = StandardCoordinatesDictionary(); coordinates_dict_instance.coordinate_x = 1
slotted_instance = SlottedCoordinates(); slotted_instance.coordinate_x = 1
print('Std dict memory:', sys.getsizeof(coordinates_dict_instance.__dict__))

### 4. Accessing Slotted Attribute Descriptors
**Explanation**: Under the hood, class attributes in `__slots__` are implemented as C-level member descriptor objects stored on the class. Accessing `ClassName.attr_name` returns a descriptor that directly indexes the instance's internal struct array in C.

**Syntax**: `descriptor = ClassName.attribute_name`

In [ ]:
class SlottedCoordinates: __slots__ = ['coordinate_x']
slotted_instance = SlottedCoordinates(); slotted_instance.coordinate_x = 5
print(getattr(slotted_instance, 'coordinate_x'))

### 5. User-Readable String Representation (`__str__`)
**Explanation**: The `__str__` dunder method is invoked by `str(obj)` and `print(obj)`. Its goal is to return an intuitive, user-friendly, human-readable string representation of the object suitable for end-user interfaces or logs.

**Syntax**: `def __str__(self): return f'Transaction({self.id})'`

In [ ]:
class EnterpriseRecord:
    def __str__(self): return 'C'
print(str(EnterpriseRecord()))

### 6. Unambiguous Developer Representation (`__repr__`)
**Explanation**: The `__repr__` method is invoked by `repr(obj)` and in the interactive REPL. Its goal is to return an explicit, unambiguous developer representation, ideally an executable Python expression that could recreate the object (e.g. `Transaction(id='TX101', amount=250.0)`). If `__str__` is not defined, Python falls back to `__repr__`.

**Syntax**: `def __repr__(self): return f'Transaction(id={self.id!r}, amount={self.amount})'`

In [ ]:
class EnterpriseRecord:
    def __repr__(self): return 'C()'
print(repr(EnterpriseRecord()))

### 7. Arithmetic Operator Overloading (`__add__` & `__radd__`)
**Explanation**: Dunder methods allow custom classes to overload arithmetic operators: `obj1 + obj2` invokes `obj1.__add__(obj2)`. If `obj1` does not implement `__add__` (or returns `NotImplemented`), Python falls back to calling `obj2.__radd__(obj1)` (reflected/right addition), enabling graceful mixed-type arithmetic.

**Syntax**: `def __add__(self, other): return Money(self.amount + other.amount)`

In [ ]:
class CashValue:
    def __init__(self, val): self.val = val
    def __add__(self, other): return CashValue(self.val + other.val)
print((CashValue(10) + CashValue(20)).val)

### 8. Operator Overloading Subtraction (`__sub__`)
**Explanation**: Overloading subtraction `obj1 - obj2` invokes `__sub__`. In financial domain models, overloading arithmetic dunders enforces currency safety (e.g. preventing adding USD to EUR without explicit currency conversion).

**Syntax**: `def __sub__(self, other): return Money(self.amount - other.amount)`

In [ ]:
class CashValue:
    def __init__(self, val): self.val = val
    def __sub__(self, other): return CashValue(self.val - other.val)
print((CashValue(50) - CashValue(20)).val)

### 9. Value Equality Protocol (`__eq__`)
**Explanation**: The `==` operator invokes `__eq__`. By default, user-defined classes inherit `object.__eq__`, which compares object identity `self is other`. Overriding `__eq__` allows comparing domain fields. CRITICAL INTERVIEW RULE: If you override `__eq__`, Python sets `__hash__ = None` by default, making instances unhashable unless you also implement `__hash__`.

**Syntax**: `def __eq__(self, other): return isinstance(other, self.__class__) and self.id == other.id`

In [ ]:
class CashValue:
    def __init__(self, val): self.val = val
    def __eq__(self, other): return self.val == other.val
print(CashValue(10) == CashValue(10))

### 10. Rich Comparisons & Ordering (`__lt__`, `functools.total_ordering`)
**Explanation**: Rich comparison dunders include `__lt__` (`<`), `__le__` (`<=`), `__gt__` (`>`), and `__ge__` (`>=`). Writing all 6 comparison methods is tedious; decorating the class with `@functools.total_ordering` requires implementing only `__eq__` and one ordering method (e.g. `__lt__`), and auto-generates the remaining comparisons.

**Syntax**: `from functools import total_ordering; @total_ordering class Order: ...`

In [ ]:
class CashValue:
    def __init__(self, val): self.val = val
    def __lt__(self, other): return self.val < other.val
print(CashValue(10) < CashValue(20))

### 11. Callable Object Instances (`__call__`)
**Explanation**: Defining `__call__(self, *args, **kwargs)` makes an instance callable like a standard function `obj(*args)`. This is widely used for creating stateful decorators, policy evaluators, and factory pipelines.

**Syntax**: `def __call__(self, *args): return self.evaluate(*args)`

In [ ]:
class CallableValidator:
    def __call__(self): return 'Called'
print(CallableValidator()())

### 12. Container Protocol: Sequence Access (`__getitem__`, `__len__`)
**Explanation**: Implementing `__getitem__(self, key)` and `__len__(self)` allows custom classes to behave like lists or mappings (`obj[index]`, `len(obj)`). Implementing `__getitem__` also automatically gives the object basic `for item in obj:` iteration support.

**Syntax**: `def __getitem__(self, index): return self._records[index]` / `def __len__(self): return len(self._records)`

In [ ]:
class ContainerClass:
    def __getitem__(self, index): return index * 2
print(ContainerClass()[5])

### 13. Container Protocol: Membership Checking (`__contains__`)
**Explanation**: The `item in obj` syntax invokes `__contains__(self, item)`. If `__contains__` is not implemented, Python falls back to iterating through the sequence via `__iter__` or `__getitem__` in O(N) time. Implementing `__contains__` with internal hash lookups provides O(1) membership checks.

**Syntax**: `def __contains__(self, key): return key in self._lookup_set`

In [ ]:
class ContainerClass:
    def __contains__(self, val): return val == 'OK'
print('OK' in ContainerClass())

### 14. Custom Hashing Protocol (`__hash__`)
**Explanation**: To allow instances to be stored in sets or used as dictionary keys, implement `__hash__(self)` returning a fixed integer. Invariant rule: If two objects are equal according to `__eq__`, they MUST produce the exact same `__hash__` value. The hashed fields must be immutable.

**Syntax**: `def __hash__(self): return hash((self.id, self.timestamp))`

In [ ]:
class HashableWrapper:
    def __init__(self, val): self.val = val
    def __hash__(self): return hash(self.val)
    def __eq__(self, other): return self.val == other.val
print(hash(HashableWrapper(10)))

### 15. Truth Value Testing Protocol (`__bool__`)
**Explanation**: When an instance is evaluated in a boolean context `if obj:`, Python calls `__bool__()`. If `__bool__` is not defined, Python queries `__len__()` (truthy if > 0). If neither is defined, instances evaluate to `True` by default.

**Syntax**: `def __bool__(self): return self.amount > 0`

In [ ]:
class FalsyOverriddenClass:
    def __bool__(self): return False
if not FalsyOverriddenClass(): print('F is Falsy!')

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Building custom immutable transaction containers with fast slotted memory, container dunder indexing, and rich comparison sorting.


In [ ]:
# Solution:
class CashFlow:
    __slots__ = ['amount']
    def __init__(self, amt): self.amount = amt
    def __add__(self, o):
        return CashFlow(self.amount + o.amount)
    def __eq__(self, o):
        return self.amount == o.amount

cf1 = CashFlow(100.0)
cf2 = CashFlow(150.0)
print('Add flow:', (cf1 + cf2).amount)


### Q2: Custom Container Ledger Protocol
**Explanation**: **Scenario**: Implement a `TransactionLedger` container supporting `len()`, `ledger[tx_id]` dictionary-style lookups, and `in` membership testing across parsed transaction rows.

**Syntax**: `class TransactionLedger: def __getitem__(self, key): ...`

In [ ]:
# Solution:
class TxCollection:
    def __init__(self):
        self.txs = ['TX1', 'TX2', 'TX3']
    def __getitem__(self, idx): return self.txs[idx]
    def __contains__(self, val): return val in self.txs

col = TxCollection()
print('First element:', col[0], '| Has TX2?:', 'TX2' in col)
